# 06 - Extract shadow variants from compact aggregation bundles

This notebook converts compact aggregation bundles into easier-to-read thematic rasters.

It generalizes the original monthly-only script so that it can process:
- `global_core_stats.tif`
- `global_hourly_core_stats.tif`
- `global_timeband_core_stats.tif`
- `monthly_bundle_YYYYMM.tif`

For every detected temporal prefix ending with `valid_count`, it derives four 1-band rasters:
- **GROUND**: open-ground shadow frequency
- **FOOTPRINT**: binary structure footprint
- **TOTAL**: total pedestrian shadow frequency
- **ROOF_SURFACE**: high-surface shadow frequency on structures

The workflow requires a `static_masks.tif` aligned to the compact bundles.


## 1. Install the required libraries

Run this only if the current environment does not already contain the required dependencies.


In [ ]:
# %pip install rasterio numpy


## 2. Import the libraries used by the extraction workflow

These imports cover:
- raster reading and writing
- windowed processing
- mask warping and alignment
- file discovery and naming


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re
import warnings

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window


## 3. Configure the extraction task

Set:
- the folder containing the compact bundles
- the aligned `static_masks.tif`
- the output folder
- the processing block size
- the list of variants to derive

This generalized workflow scans all relevant aggregation bundles automatically.


In [ ]:
warnings.filterwarnings("ignore", category=RuntimeWarning)

@dataclass
class VariantExtractionConfig:
    raster_dir: str = "."
    static_masks: str = "static_masks.tif"
    output_dir: str = "output_shadow_variants"
    block_size: int = 2048

    variants: tuple[str, ...] = ("GROUND", "FOOTPRINT", "TOTAL", "ROOF_SURFACE")

cfg = VariantExtractionConfig()
cfg


## 4. Generate raster windows for block-wise processing

Large rasters are processed window by window to keep memory usage under control.


In [ ]:
def generate_windows(width: int, height: int, block_size: int = 2048):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            w = min(block_size, width - col_off)
            h = min(block_size, height - row_off)
            yield Window(col_off, row_off, w, h)


## 5. Discover compact bundles and classify them

This generalized discovery step recognizes:
- `global_core_stats.tif`
- `global_hourly_core_stats.tif`
- `global_timeband_core_stats.tif`
- `monthly_bundle_YYYYMM.tif`

Each bundle is assigned:
- a **scope key** (`global` or `YYYYMM`)
- a **bundle kind**


In [ ]:
@dataclass(frozen=True)
class BundleSpec:
    filename: Path
    scope_key: str
    bundle_kind: str
    label: str


def discover_compact_bundles(raster_dir: Path) -> list[BundleSpec]:
    specs: list[BundleSpec] = []

    for tif in sorted(raster_dir.glob("*.tif")):
        name = tif.name

        if name == "static_masks.tif":
            continue

        if name == "global_core_stats.tif":
            specs.append(BundleSpec(tif, "global", "global_core", "Global core stats"))
            continue

        if name == "global_hourly_core_stats.tif":
            specs.append(BundleSpec(tif, "global", "global_hourly", "Global hourly core stats"))
            continue

        if name == "global_timeband_core_stats.tif":
            specs.append(BundleSpec(tif, "global", "global_timeband", "Global time-band core stats"))
            continue

        m = re.match(r"monthly_bundle_(\d{6})\.tif$", name, flags=re.IGNORECASE)
        if m:
            yyyymm = m.group(1)
            specs.append(BundleSpec(tif, yyyymm, "monthly", f"Monthly bundle {yyyymm}"))
            continue

    return specs


bundle_specs = discover_compact_bundles(Path(cfg.raster_dir).resolve())
print(f"Bundles found: {len(bundle_specs)}")
bundle_specs


## 6. Normalize temporal prefixes and bundle naming

Each compact bundle contains bands named like:
- `valid_count`
- `h06_valid_count`
- `early_morning_valid_count`
- `month_valid_count`

This block:
- finds all valid-count prefixes
- converts them into a clean output key
- produces a stable output raster naming scheme


In [ ]:
def derive_prefixes(bundle_src: rasterio.io.DatasetReader) -> list[str]:
    names = list(bundle_src.descriptions)
    prefixes = [n[:-len("valid_count")] for n in names if n and n.endswith("valid_count")]
    return prefixes


def normalize_prefix(prefix: str, bundle_kind: str) -> str:
    raw = prefix.rstrip("_")
    if not raw:
        if bundle_kind == "global_core":
            return "fullperiod"
        return "fullperiod"
    return raw.replace("_", "").strip().lower()


def structure_band_index(mask_src: rasterio.io.DatasetReader) -> int:
    descs = tuple(d or "" for d in mask_src.descriptions)
    for i, d in enumerate(descs, start=1):
        if d.strip().lower() == "structure_mask":
            return i
    return 1


def band_index(bundle_src: rasterio.io.DatasetReader, band_name: str) -> int:
    names = list(bundle_src.descriptions)
    if band_name not in names:
        raise KeyError(f"Band '{band_name}' not found in {bundle_src.name}")
    return names.index(band_name) + 1


## 7. Build output filenames

Output rasters follow a unified convention:

`ombra_<scope>_<prefix>_<VARIANT>.tif`

Examples:
- `ombra_global_fullperiod_TOTAL.tif`
- `ombra_global_h06_TOTAL.tif`
- `ombra_global_earlymorning_GROUND.tif`
- `ombra_202506_month_TOTAL.tif`


In [ ]:
def build_output_path(output_dir: Path, scope_key: str, prefix_clean: str, variant: str) -> Path:
    return output_dir / f"ombra_{scope_key}_{prefix_clean}_{variant}.tif"


## 8. Derive thematic rasters from one compact bundle

For each temporal prefix, the logic is:

- `freq_G = count_G / valid_count`
- `freq_S = count_S / valid_count`
- `GROUND = freq_G` only outside structures
- `FOOTPRINT = 1` only on structures
- `TOTAL = 1` on structures, otherwise `freq_G`
- `ROOF_SURFACE = freq_S` only on structures

This function is generalized across global and monthly bundle types.


In [ ]:
def process_bundle(spec: BundleSpec, masks_path: Path, output_dir: Path, block_size: int = 2048, variants: tuple[str, ...] = ("GROUND", "FOOTPRINT", "TOTAL", "ROOF_SURFACE")):
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing bundle: {spec.filename.name}")
    print(f"  Scope key:   {spec.scope_key}")
    print(f"  Bundle kind: {spec.bundle_kind}")

    with rasterio.open(spec.filename) as src_bundle:
        profile_base = src_bundle.profile.copy()
        width, height = src_bundle.width, src_bundle.height
        prefixes = derive_prefixes(src_bundle)

        if not prefixes:
            raise RuntimeError(f"No '*valid_count' bands found in {spec.filename.name}")

        print(f"  Prefixes found: {prefixes}")

        with rasterio.open(masks_path) as src_mask:
            mask_band = structure_band_index(src_mask)

            with WarpedVRT(
                src_mask,
                crs=src_bundle.crs,
                transform=src_bundle.transform,
                width=width,
                height=height,
                resampling=Resampling.nearest,
            ) as vrt_mask:

                open_outputs: dict[str, dict[str, rasterio.io.DatasetWriter]] = {}

                try:
                    for prefix in prefixes:
                        prefix_clean = normalize_prefix(prefix, spec.bundle_kind)
                        open_outputs[prefix] = {}

                        for variant in variants:
                            out_profile = profile_base.copy()
                            out_profile.update(
                                count=1,
                                dtype=rasterio.float32,
                                nodata=np.nan,
                                tiled=True,
                                blockxsize=512,
                                blockysize=512,
                                compress="lzw",
                                BIGTIFF="IF_SAFER",
                            )
                            out_name = build_output_path(output_dir, spec.scope_key, prefix_clean, variant)
                            dst = rasterio.open(out_name, "w", **out_profile)
                            dst.set_band_description(1, f"{variant} frequency - {spec.scope_key} - {prefix_clean}")
                            dst.update_tags(
                                product=variant,
                                scope_key=spec.scope_key,
                                bundle_kind=spec.bundle_kind,
                                source_bundle=spec.filename.name,
                                prefix=prefix,
                                prefix_clean=prefix_clean,
                                logic=(
                                    "GROUND=freq_G outside structures; "
                                    "FOOTPRINT=1 on structures; "
                                    "TOTAL=1 on structures else freq_G; "
                                    "ROOF_SURFACE=freq_S on structures"
                                ),
                            )
                            open_outputs[prefix][variant] = dst

                    windows = list(generate_windows(width, height, block_size=block_size))
                    print(f"  Processing {len(windows)} windows...")

                    for wi, window in enumerate(windows, start=1):
                        if wi == 1 or wi % 20 == 0 or wi == len(windows):
                            print(f"    window {wi}/{len(windows)}")

                        structures = vrt_mask.read(mask_band, window=window) > 0

                        for prefix in prefixes:
                            idx_valid = band_index(src_bundle, f"{prefix}valid_count")
                            idx_g = band_index(src_bundle, f"{prefix}count_G")
                            idx_s = band_index(src_bundle, f"{prefix}count_S")

                            valid_count = src_bundle.read(idx_valid, window=window).astype(np.float32)
                            count_g = src_bundle.read(idx_g, window=window).astype(np.float32)
                            count_s = src_bundle.read(idx_s, window=window).astype(np.float32)

                            if not np.any(valid_count > 0):
                                empty = np.full(valid_count.shape, np.nan, dtype=np.float32)
                                for variant in variants:
                                    open_outputs[prefix][variant].write(empty, 1, window=window)
                                continue

                            with np.errstate(divide="ignore", invalid="ignore"):
                                freq_g = np.where(valid_count > 0, count_g / valid_count, np.nan).astype(np.float32)
                                freq_s = np.where(valid_count > 0, count_s / valid_count, np.nan).astype(np.float32)

                            ground = np.where(~structures, freq_g, np.nan).astype(np.float32)
                            footprint = np.where(structures, 1.0, np.nan).astype(np.float32)
                            total = np.where(structures, 1.0, freq_g).astype(np.float32)
                            roof_surface = np.where(structures, freq_s, np.nan).astype(np.float32)

                            if "GROUND" in variants:
                                open_outputs[prefix]["GROUND"].write(ground, 1, window=window)
                            if "FOOTPRINT" in variants:
                                open_outputs[prefix]["FOOTPRINT"].write(footprint, 1, window=window)
                            if "TOTAL" in variants:
                                open_outputs[prefix]["TOTAL"].write(total, 1, window=window)
                            if "ROOF_SURFACE" in variants:
                                open_outputs[prefix]["ROOF_SURFACE"].write(roof_surface, 1, window=window)

                finally:
                    for per_prefix in open_outputs.values():
                        for dst in per_prefix.values():
                            try:
                                dst.close()
                            except Exception:
                                pass

    print(f"  Done: {spec.filename.name}")


## 9. Validate inputs before processing

This check confirms that:
- at least one compact bundle was found
- `static_masks.tif` exists


In [ ]:
bundle_specs = discover_compact_bundles(Path(cfg.raster_dir).resolve())
if not bundle_specs:
    raise FileNotFoundError(f"No supported compact bundles found in {Path(cfg.raster_dir).resolve()}")

static_masks_path = Path(cfg.static_masks).resolve()
if not static_masks_path.exists():
    raise FileNotFoundError(f"static_masks.tif not found: {static_masks_path}")

print(f"Supported bundles found: {len(bundle_specs)}")
for spec in bundle_specs:
    print(f"  - {spec.filename.name} ({spec.bundle_kind})")


## 10. Run the generalized extraction workflow

This cell processes every discovered compact bundle and writes the derived shadow variants to the output directory.


In [ ]:
output_dir = Path(cfg.output_dir).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

for spec in bundle_specs:
    process_bundle(
        spec=spec,
        masks_path=static_masks_path,
        output_dir=output_dir,
        block_size=cfg.block_size,
        variants=cfg.variants,
    )

print(f"\nAll done. Outputs written to: {output_dir}")


## 11. Optional quick inventory of the generated rasters

This is a simple post-run check that lists the output files created by the notebook.


In [ ]:
generated = sorted(Path(cfg.output_dir).resolve().glob("ombra_*.tif"))
print(f"Generated rasters: {len(generated)}")
generated[:20]
